In [1]:
import numpy as np
import pandas as pd
import os
import cv2
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, GlobalAveragePooling2D, Dropout,BatchNormalization, MaxPooling2D
from tensorflow.keras.optimizers import Adam


In [2]:
def image_import(path):
    Data=[]
    Labels=[]
    Categories=['cat','dog']
    for cate in Categories:
        folder=os.path.join(path,cate)
        label=Categories.index(cate)
        for img in os.listdir(folder):
            img_path=os.path.join(folder,img)
            image=cv2.imread(img_path)
            if image is None:
                continue
            image=cv2.resize(image,(128,128))
            Data.append(image)
            Labels.append(label)
    return np.array(Data),np.array(Labels)        


In [3]:
train_data='train'
val_data='val'


In [4]:
X_train,y_train=image_import(train_data)
X_val,y_val=image_import(val_data)


In [5]:
X_train = X_train/255.0
X_val = X_val/255.0


train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

Train_Datagen = train_datagen.flow(
    X_train, y_train,
    batch_size=32,
    shuffle=True
)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator()

Val_data = val_datagen.flow(
    X_val, y_val,
    batch_size=32,
    shuffle=False
)

In [6]:
print(np.bincount(y_train))
print(np.bincount(y_val))

[ 95 180]
[24 46]


In [7]:



model = Sequential([

    
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.3),

    
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.5),

    
    Flatten(),
    
    Dense(512, activation='relu'),
    BatchNormalization(),   
    Dropout(0.5),

    Dense(256, activation='relu'),
    BatchNormalization(),   
    Dropout(0.5),

    Dense(128, activation='relu'),
    BatchNormalization(), 
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])
model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 batch_normalization (Batch  (None, 128, 128, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 64, 64, 32)        0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 64, 64, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 64)        18496     
                                                                 
 batch_normalization_1 (Bat  (None, 64, 64, 64)       

In [8]:
from sklearn.utils import  class_weight
class_weights=class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)


class_weights_dict = dict(enumerate(class_weights))


In [9]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7) 
model.fit(
    Train_Datagen,
    epochs=25,
    verbose=1,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights_dict,
    validation_data=Val_data,
    shuffle=True
)


Epoch 1/25


9/9 [==============================] - 9s 568ms/step - loss: 0.9949 - accuracy: 0.5055 - val_loss: 0.6774 - val_accuracy: 0.6143 - lr: 0.0010
Epoch 2/25
9/9 [==============================] - 4s 435ms/step - loss: 1.0674 - accuracy: 0.5164 - val_loss: 0.6738 - val_accuracy: 0.6429 - lr: 0.0010
Epoch 3/25
9/9 [==============================] - 6s 592ms/step - loss: 0.9368 - accuracy: 0.5091 - val_loss: 0.6543 - val_accuracy: 0.6429 - lr: 0.0010
Epoch 4/25
9/9 [==============================] - 4s 431ms/step - loss: 0.9954 - accuracy: 0.5418 - val_loss: 0.7420 - val_accuracy: 0.3000 - lr: 0.0010
Epoch 5/25
9/9 [==============================] - 5s 471ms/step - loss: 0.8681 - accuracy: 0.5418 - val_loss: 0.7253 - val_accuracy: 0.3714 - lr: 0.0010
Epoch 6/25
9/9 [==============================] - 6s 653ms/step - loss: 0.9532 - accuracy: 0.5309 - val_loss: 0.7803 - val_accuracy: 0.4143 - lr: 0.0010
Epoch 7/25
9/9 [==============================] - 3s 352ms/step - loss: 0.8350 -

In [10]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = (model.predict(X_val) > 0.5).astype(int)
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

3/3 [==============================] - 0s 50ms/step
[[ 1 23]
 [ 2 44]]
              precision    recall  f1-score   support

           0       0.33      0.04      0.07        24
           1       0.66      0.96      0.78        46

    accuracy                           0.64        70
   macro avg       0.50      0.50      0.43        70
weighted avg       0.55      0.64      0.54        70

